# 02 — Preprocesamiento y Feature Engineering
## Aircraft Engine Predictive Maintenance · NASA C-MAPSS Dataset

**Objetivo de este notebook:** limpiar y preparar los 4 subconjuntos del dataset.

Lo que haremos:
1. Cargar los 4 subconjuntos (FD001, FD002, FD003, FD004)
2. Normalizar por condición operativa (crítico para FD002 y FD004)
3. Eliminar sensores inútiles
4. Crear features de medias móviles
5. Crear la variable objetivo (TARGET)
6. Guardar los datos procesados

---

## 1. Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

print('Librerías cargadas correctamente ✅')

## 2. Cargar los 4 subconjuntos

Cada subconjunto tiene distinta complejidad:
- FD001: 1 condición operativa, 1 tipo de fallo
- FD002: 6 condiciones operativas, 1 tipo de fallo
- FD003: 1 condición operativa, 2 tipos de fallo
- FD004: 6 condiciones operativas, 2 tipos de fallo

In [ ]:
columnas = [
    'motor_id', 'ciclo',
    'setting_1', 'setting_2', 'setting_3',
    's1', 's2', 's3', 's4', 's5',
    's6', 's7', 's8', 's9', 's10',
    's11', 's12', 's13', 's14', 's15',
    's16', 's17', 's18', 's19', 's20', 's21'
]

def cargar_dataset(nombre):
    """Carga un subconjunto y añade una columna con su nombre."""
    df = pd.read_csv(
        f'../data/raw/train_{nombre}.txt',
        sep=' ', header=None, names=columnas, index_col=False
    )
    df = df.dropna(axis=1, how='all')
    df['subconjunto'] = nombre
    # Hacer motor_id único entre subconjuntos
    offset = {'FD001': 0, 'FD002': 1000, 'FD003': 2000, 'FD004': 3000}
    df['motor_id'] = df['motor_id'] + offset[nombre]
    return df

# Cargar los 4 subconjuntos
fd001 = cargar_dataset('FD001')
fd002 = cargar_dataset('FD002')
fd003 = cargar_dataset('FD003')
fd004 = cargar_dataset('FD004')

# Combinar en un solo dataframe
df = pd.concat([fd001, fd002, fd003, fd004], ignore_index=True)

print(f'FD001: {len(fd001):,} registros · {fd001["motor_id"].nunique()} motores')
print(f'FD002: {len(fd002):,} registros · {fd002["motor_id"].nunique()} motores')
print(f'FD003: {len(fd003):,} registros · {fd003["motor_id"].nunique()} motores')
print(f'FD004: {len(fd004):,} registros · {fd004["motor_id"].nunique()} motores')
print(f'\nTotal combinado: {len(df):,} registros · {df["motor_id"].nunique()} motores')

## 3. Calcular el RUL y crear el TARGET

In [ ]:
# Calcular RUL
ciclo_max = df.groupby('motor_id')['ciclo'].max().reset_index()
ciclo_max.columns = ['motor_id', 'ciclo_max']
df = df.merge(ciclo_max, on='motor_id')
df['RUL'] = df['ciclo_max'] - df['ciclo']
df = df.drop('ciclo_max', axis=1)

# Crear TARGET
UMBRAL_RIESGO = 30
df['target'] = (df['RUL'] < UMBRAL_RIESGO).astype(int)

print(f'RUL calculado ✅')
print(f'Rango de RUL: {df["RUL"].min()} — {df["RUL"].max()} ciclos')
print(f'\nDistribución del target:')
print(f'  Seguro  (0): {(df["target"]==0).sum():,} ({(df["target"]==0).mean()*100:.1f}%)')
print(f'  En riesgo (1): {(df["target"]==1).sum():,} ({(df["target"]==1).mean()*100:.1f}%)')

## 4. Normalización por condición operativa

**Este es el paso más importante al combinar los 4 subconjuntos.**

FD002 y FD004 tienen 6 condiciones operativas distintas (altitud, velocidad, temperatura).
Un sensor que marca 600 en condición 1 puede marcar 800 en condición 3 — no por degradación,
sino porque el avión está volando en condiciones distintas.

Si no normalizamos por condición, el modelo aprende las diferencias entre condiciones
en lugar de aprender la degradación del motor.

**Solución:** usamos KMeans para identificar automáticamente las condiciones operativas
y luego normalizamos cada sensor dentro de cada condición.

In [ ]:
# Identificar condiciones operativas con KMeans
# Las settings definen la condición operativa
settings = df[['setting_1', 'setting_2', 'setting_3']].copy()

# Usamos 6 clusters — el máximo de condiciones que hay en FD002/FD004
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
df['condicion'] = kmeans.fit_predict(settings)

print(f'Condiciones operativas identificadas: {df["condicion"].nunique()}')
print(f'\nDistribución por condición:')
print(df['condicion'].value_counts().sort_index())
print(f'\nDistribución por subconjunto y condición:')
print(df.groupby(['subconjunto', 'condicion']).size().unstack(fill_value=0))

In [ ]:
# Normalizar cada sensor dentro de cada condición operativa
sensores_utiles = ['s2', 's3', 's4', 's7', 's8', 's9',
                   's11', 's12', 's13', 's14', 's15',
                   's17', 's20', 's21']

for sensor in sensores_utiles:
    df[f'{sensor}_norm'] = df.groupby('condicion')[sensor].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-8)
    )

print(f'Normalización por condición completada ✅')
print(f'Sensores normalizados: {len(sensores_utiles)}')
print(f'\nEjemplo — s11_norm:')
print(f'  Media: {df["s11_norm"].mean():.4f} (debería ser ~0)')
print(f'  Std:   {df["s11_norm"].std():.4f} (debería ser ~1)')

## 5. Crear features de medias móviles

Igual que en el notebook anterior, creamos medias móviles de 10 ciclos
para capturar la tendencia de degradación de cada sensor.

In [ ]:
VENTANA = 10

cols_norm = [f'{s}_norm' for s in sensores_utiles]

for col in cols_norm:
    df[f'{col}_mm'] = (
        df.groupby('motor_id')[col]
        .transform(lambda x: x.rolling(window=VENTANA, min_periods=1).mean())
    )

print(f'Medias móviles creadas ✅')
print(f'Total features: {len(cols_norm)} normalizadas + {len(cols_norm)} medias móviles = {len(cols_norm)*2}')

## 6. Preparar datos para el modelo

In [ ]:
from sklearn.model_selection import train_test_split

# Features: sensores normalizados + medias móviles
feature_cols = cols_norm + [f'{c}_mm' for c in cols_norm]

X = df[feature_cols]
y = df['target']

# Split estratificado 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Registros de entrenamiento: {X_train.shape[0]:,}')
print(f'Registros de test:          {X_test.shape[0]:,}')
print(f'Features totales:           {X_train.shape[1]}')
print(f'\nDistribución target en train:')
print(f'  Seguro (0):    {(y_train==0).sum():,} ({(y_train==0).mean()*100:.1f}%)')
print(f'  En riesgo (1): {(y_train==1).sum():,} ({(y_train==1).mean()*100:.1f}%)')

## 7. Guardar los datos procesados

In [ ]:
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print('Datos guardados en data/processed/ ✅')
print('\nArchivos creados:')
print('  X_train.csv — features de entrenamiento')
print('  X_test.csv  — features de test')
print('  y_train.csv — target de entrenamiento')
print('  y_test.csv  — target de test')

## 8. Resumen del preprocesamiento

In [ ]:
print('=' * 60)
print('RESUMEN DEL PREPROCESAMIENTO — 4 SUBCONJUNTOS')
print('=' * 60)
print(f'Subconjuntos combinados:    FD001 + FD002 + FD003 + FD004')
print(f'Total motores:              {df["motor_id"].nunique()}')
print(f'Total registros:            {len(df):,}')
print(f'Condiciones operativas:     6 (identificadas con KMeans)')
print(f'Sensores útiles:            {len(sensores_utiles)}')
print(f'Features para modelo:       {X_train.shape[1]}')
print(f'Registros entrenamiento:    {X_train.shape[0]:,}')
print(f'Registros test:             {X_test.shape[0]:,}')
print()
print('MEJORA vs solo FD001:')
print(f'  Motores: 100 → {df["motor_id"].nunique()} (+{df["motor_id"].nunique()-100})')
print(f'  Registros: 20,631 → {len(df):,} (+{len(df)-20631:,})')
print(f'  Condiciones: 1 → 6 (modelo más robusto)')
print('=' * 60)

---
**Notebook completado ✅**  
Siguiente paso: `03_modelado.ipynb`